### Setup

In [1]:
import sqlite3
conn = sqlite3.connect("students.db")
cursor = conn.cursor()

# 1. Re-create the STUDENTS table [cite: 81]
cursor.execute("""
CREATE TABLE IF NOT EXISTS students (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    full_name TEXT NOT NULL,
    course TEXT NOT NULL,
    year_level INTEGER NOT NULL
)""")

# 2. Re-create the COURSES table [cite: 242]
cursor.execute("""
CREATE TABLE IF NOT EXISTS courses (
    course_code TEXT PRIMARY KEY,
    description TEXT NOT NULL
)""")

# 3. Insert data for both tables [cite: 103, 248]
students_data = [("Anna Reyes", "BS ECE", 3), ("Carlos Dela Cruz", "BS EE", 2), ("Mika Santos", "BS CE", 1)]
course_data = [
    ("BS ECE", "Electronics and Communications Engineering"),
    ("BS EE", "Electrical Engineering"),
    ("BS COE", "Computer Engineering"),
    ("BS CE", "Civil Engineering")
]

cursor.executemany("INSERT OR IGNORE INTO students (full_name, course, year_level) VALUES (?, ?, ?)", students_data) 
cursor.executemany("INSERT OR IGNORE INTO courses (course_code, description) VALUES (?, ?)", course_data)
conn.commit() 

print("Database, Students table, and Courses table are all ready!")

Database, Students table, and Courses table are all ready!


### 1. Highest Year Level

In [2]:
cursor.execute("SELECT full_name, course FROM students WHERE year_level = (SELECT MAX(year_level) FROM students)")
print(cursor.fetchall())

[('Anna Reyes', 'BS ECE')]


### 2. Engineering Students

In [3]:
cursor.execute("SELECT * FROM students WHERE course IN (SELECT course_code FROM courses WHERE description LIKE '%Engineering%')")
print(cursor.fetchall())

[(1, 'Anna Reyes', 'BS ECE', 3, None), (2, 'Carlos Dela Cruz', 'BS EE', 2, None), (3, 'Mika Santos', 'BS CE', 1, None)]


### 3. Courses with >1 Student

In [4]:
cursor.execute("SELECT course, COUNT(*) AS total FROM students GROUP BY course HAVING total > 1")
print(cursor.fetchall())

[]


### 4. Case Statement (Status)

In [5]:
cursor.execute("SELECT *, CASE WHEN year_level >= 4 THEN 'Graduating' ELSE 'Regular' END AS status FROM students")
print(cursor.fetchall())

[(1, 'Anna Reyes', 'BS ECE', 3, None, 'Regular'), (2, 'Carlos Dela Cruz', 'BS EE', 2, None, 'Regular'), (3, 'Mika Santos', 'BS CE', 1, None, 'Regular')]


### 5. Left Join (All Students)

In [6]:
cursor.execute("SELECT s.full_name, c.description FROM students s LEFT JOIN courses c ON s.course = c.course_code")
print(cursor.fetchall())

[('Anna Reyes', 'Electronics and Communications Engineering'), ('Carlos Dela Cruz', 'Electrical Engineering'), ('Mika Santos', 'Civil Engineering')]


### 6. Three-part Names Count

In [7]:
cursor.execute("SELECT COUNT(*) FROM students WHERE full_name LIKE '% % %'")
print(cursor.fetchone())

(1,)


### 7. Max Year Level by Course

In [8]:
cursor.execute("SELECT course, MAX(year_level) FROM students GROUP BY course ORDER BY MAX(year_level) DESC")
print(cursor.fetchall())

[('BS ECE', 3), ('BS EE', 2), ('BS CE', 1)]


### 8. Even-numbered Rows

In [9]:
cursor.execute("SELECT * FROM students WHERE rowid % 2 = 0")
print(cursor.fetchall())

[(2, 'Carlos Dela Cruz', 'BS EE', 2, None)]


### 9. Names with two 'e's (GLOB)

In [10]:
cursor.execute("SELECT full_name FROM students WHERE full_name GLOB '*e*e*'")
print(cursor.fetchall())

[('Anna Reyes',)]


### 10. Students with Missing Course Records

In [11]:
cursor.execute("SELECT full_name, course FROM students WHERE NOT EXISTS (SELECT 1 FROM courses WHERE course_code = students.course)")
print(cursor.fetchall())

[]


### 11. Course Statistics

In [12]:
cursor.execute("SELECT course, COUNT(*), AVG(year_level) FROM students GROUP BY course")
print(cursor.fetchall())

[('BS CE', 1, 1.0), ('BS ECE', 1, 3.0), ('BS EE', 1, 2.0)]


### 12. Student Initials

In [13]:
cursor.execute("SELECT full_name, SUBSTR(full_name, 1, 1) AS initial FROM students")
print(cursor.fetchall())

[('Anna Reyes', 'A'), ('Carlos Dela Cruz', 'C'), ('Mika Santos', 'M')]


### 13. Dynamic Search with Limit

In [14]:
cursor.execute("SELECT * FROM students WHERE full_name LIKE ? LIMIT 1", ('%' + input("Enter keyword: ") + '%',))
print(cursor.fetchone())

None


### 14. Most Popular Course

In [15]:
cursor.execute("SELECT course, COUNT(*) FROM students GROUP BY course ORDER BY COUNT(*) DESC LIMIT 1")
print(cursor.fetchall())

[('BS EE', 1)]


### 15. Filtering by Name and Year Range

In [16]:
cursor.execute("SELECT DISTINCT course FROM students WHERE full_name LIKE '%a%' AND year_level BETWEEN 2 AND 4")
print(cursor.fetchall())

[('BS ECE',), ('BS EE',)]


### 16. Longest Name

In [17]:
cursor.execute("SELECT * FROM students WHERE LENGTH(full_name) = (SELECT MAX(LENGTH(full_name)) FROM students)")
print(cursor.fetchall())

[(2, 'Carlos Dela Cruz', 'BS EE', 2, None)]


### 17. Lowest Year Level (BS CE Count)

In [18]:
cursor.execute("SELECT COUNT(*) FROM students WHERE course = 'BS CE' AND year_level = (SELECT MIN(year_level) FROM students WHERE course = 'BS CE')")
print(cursor.fetchall())

[(1,)]


### 18. Computer Engineering Search

In [19]:
cursor.execute("SELECT s.full_name, s.year_level, c.description FROM students s JOIN courses c ON s.course = c.course_code WHERE c.description LIKE '%Computer%'")
print(cursor.fetchall())

[]


### 19. Error Handling (Missing Table)

In [20]:
try:
    cursor.execute("SELECT * FROM unknown_table")
except Exception as e:
    print("Error:", e)

Error: no such table: unknown_table


### 20. Course with Highest Students (Tie-Safe):

In [21]:
cursor.execute("SELECT course, COUNT(*) FROM students GROUP BY course HAVING COUNT(*) = (SELECT MAX(cnt) FROM (SELECT COUNT(*) AS cnt FROM students GROUP BY course))")
print(cursor.fetchall())

[('BS CE', 1), ('BS ECE', 1), ('BS EE', 1)]
